In [ ]:
!pip install langchain langchain-community langchain-google-genai pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 8.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = ""

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model = "gemini-3.5-flash",
    temperature = 0
)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

In [ ]:
loader = PyPDFLoader("Rezume.pdf")
documents = loader.load()

In [ ]:
from langchain_core.tools import tool

In [ ]:
@tool
def read_pdf(question: str) -> str:
  """Answer the questoin using the PDF content."""
  text = "\n".join(doc.page_content for doc in documents)
  return text

In [ ]:
#create agent
agent = create_agent(
    model = llm,
    tools=[read_pdf],
    system_prompt="""
    You are a pdf reader agent,
    use the read_pdf tool to find information from the pdf and
    Answer only using information from the pdf,
    try to answer everyquestion
    """
)

In [ ]:
question = input("Enter question: ")
response = agent.invoke({
    "messages":[
        {
            "role" : "user",
            "content" : question
        }
    ]
})
print(response["messages"][-1].content)

Enter question: hi
[{'type': 'text', 'text': 'Hello! I have accessed the document, which is the resume of **Rokeshwaran M**, an Aspiring Full Stack Developer. \n\nHow can I help you with information from this resume today?', 'extras': {'signature': 'EsECCr4CAWkUfRMI5JRTD+wUdIZiMQXChCghShd6Kk2z+wd4gkWZ7s6S03jKsdfJDxrYUFOWZa1aPXWy29lmOQukDKEUK3FF+FBDL7P9Id4IxwQS1cjdMYCXN3ekif32QkvnoxSSL0cNnnduCab2VYsdZysepp8v7bCbbKh+0UCWu2m1xfchIdJReRHr9TETuHH4vcz3XgiIivkfucwTnDDn6PfYE6mYdbmOIe5fx0i4wJoKmfJP9fjRJixzX24W6QqeT92VQmDDTh8E7fW9dU0n7/eTZly0ZgEI1WBMolkFHArxh+XRi1NiQX8mLTGksSgmbRz1L5pb5yNY+SEnVV7H75pQceC5XmGuNIRkKfJaW0/aCseumOtikk3TAWE/UHw6J6xQlgTfmB4F7YRKnSG6dTIoB0I5l4HZC+mmTz5KH9r/'}}]
